In [1]:
import pandas as pd
import mysql.connector
from pathlib import Path

# 1) 파일 경로
file_path = "C:/skn29/PYTHON/project/data/잔가율.xlsx"

# 2) 엑셀 읽기
raw_df = pd.read_excel(file_path)

# 3) 실제 데이터 구간만 사용
data_df = raw_df.iloc[1:].copy()

# 4) 국산-승용
korean_passenger = data_df[['국산-승용', 'Unnamed: 2']].copy()
korean_passenger.columns = ['model_year', 'residual_value_rate']
korean_passenger['origin_type'] = '국산'
korean_passenger['body_type'] = '승용'

# 5) 외산-승용
imported_passenger = data_df[['외산-승용', 'Unnamed: 4']].copy()
imported_passenger.columns = ['model_year', 'residual_value_rate']
imported_passenger['origin_type'] = '외산'
imported_passenger['body_type'] = '승용'

# 6) 비영업용 승합 -> 국산 승합 / 외산 승합 둘 다 복제
van_base = data_df[['비영업용 승합', 'Unnamed: 6']].copy()
van_base.columns = ['model_year', 'residual_value_rate']

korean_van = van_base.copy()
korean_van['origin_type'] = '국산'
korean_van['body_type'] = '승합'

imported_van = van_base.copy()
imported_van['origin_type'] = '외산'
imported_van['body_type'] = '승합'

# 7) 합치기
df = pd.concat(
    [korean_passenger, imported_passenger, korean_van, imported_van],
    ignore_index=True
)

# 8) 전처리
df['model_year'] = pd.to_numeric(df['model_year'], errors='coerce')
df['residual_value_rate'] = pd.to_numeric(df['residual_value_rate'], errors='coerce')

df = df.dropna(subset=['origin_type', 'body_type', 'model_year', 'residual_value_rate'])

df['model_year'] = df['model_year'].astype(int)
df['residual_value_rate'] = df['residual_value_rate'].astype(float)

# 중복 제거
df = df.drop_duplicates(subset=['origin_type', 'body_type', 'model_year'])

print("적재 대상 행 수:", len(df))
print(df.head(10))
print(df.tail(10))

# 9) MySQL 연결
conn = mysql.connector.connect(
    host='localhost',
    user='root',
    password='!didwjdgus16',
    database='car_insurance_final',
    charset='utf8mb4'
)
cursor = conn.cursor()

# 10) INSERT SQL
insert_sql = """
INSERT INTO car_value_factor (
    origin_type,
    body_type,
    model_year,
    residual_value_rate
) VALUES (%s, %s, %s, %s)
ON DUPLICATE KEY UPDATE
    residual_value_rate = VALUES(residual_value_rate),
    updated_at = CURRENT_TIMESTAMP
"""

# 11) executemany용 데이터 생성
data_to_insert = [
    (
        row.origin_type,
        row.body_type,
        row.model_year,
        row.residual_value_rate
    )
    for row in df.itertuples(index=False)
]

# 12) INSERT 실행
cursor.executemany(insert_sql, data_to_insert)
conn.commit()

print(f"{cursor.rowcount}건 처리 완료")

# 13) 확인
cursor.execute("""
SELECT origin_type, body_type, model_year, residual_value_rate
FROM car_value_factor
ORDER BY body_type, origin_type, model_year DESC
LIMIT 30
""")

for row in cursor.fetchall():
    print(row)

cursor.close()
conn.close()

적재 대상 행 수: 124
   model_year  residual_value_rate origin_type body_type
0        2025                0.826          국산        승용
1        2024                0.725          국산        승용
2        2023                0.614          국산        승용
3        2022                0.518          국산        승용
4        2021                0.437          국산        승용
5        2020                0.368          국산        승용
6        2019                0.311          국산        승용
7        2018                0.262          국산        승용
8        2017                0.221          국산        승용
9        2016                0.186          국산        승용
     model_year  residual_value_rate origin_type body_type
114        2004                 0.05          외산        승합
115        2003                 0.05          외산        승합
116        2002                 0.05          외산        승합
117        2001                 0.05          외산        승합
118        2000                 0.05          외산        승합
119 